# Notebook 4: Programming a leak valve through its RS232 port

*Python for Instrumentation and Laboratory Experiments, Pr. T. Angot*

Control the VAT Series 590 leak valve through RS232: test the device status, check the valve position, set a chosen position, hold it for a certain time, and close it.

The valve was not available, so the program could not be run.

## Functioning of the valve

- A valve plate, moved by a stepper motor, changes the conductance of the opening and so the gas flow into the vacuum chamber. Full stroke = 100 000 steps (0 = closed, 100 000 = open), about 3.5 s from closed to open.
- Two modes: **position control** (we give a position) and **pressure control** (the controller reads a gauge on its SENSOR input and moves the plate to reach a pressure setpoint).
- RS232 commands are only accepted in **remote** operation. The interlock inputs OPEN / CLOSE have priority over RS232.
- At power up the valve synchronizes (position unknown, `A:` returns 999999), then closes. On power failure it stays where it is.

## Communication

9600 baud, **7 data bits, even parity**, 1 stop bit. Commands are case sensitive and end with `\r\n`. The valve acknowledges each command within 10 ms (wait for it before the next one) or answers an error `E:xxxxxx`.

| Command | Function | Answer |
|---|---|---|
| `i:30` | status | `i:30abcdefgh`, a = access (0 local, 1 remote), b = mode (2 position control, 3 closed, 4 open, 6 hold, 8/9 interlock, D safety, E fatal error), d = warning |
| `A:` | position | `A:aaaaaa` |
| `R:aaaaaa` | go to position aaaaaa | `R:` |
| `H:` | hold the current position | `H:` |
| `C:` | close | `C:` |

In [1]:
import time
import serial
import serial.tools.list_ports as port_list

## Program

In [2]:
VALVE_SETTINGS = dict(baudrate=9600, bytesize=serial.SEVENBITS, parity=serial.PARITY_EVEN,
                      stopbits=serial.STOPBITS_ONE, timeout=1)

ACCESS_MODES = {'0': 'local', '1': 'remote', '2': 'locked remote'}
CONTROL_MODES = {'0': 'initialization', '1': 'synchronization', '2': 'position control', '3': 'closed',
                 '4': 'open', '5': 'pressure control', '6': 'hold', '8': 'interlock open',
                 '9': 'interlock closed', 'C': 'power failure', 'D': 'safety mode', 'E': 'fatal error'}
ERRORS = {'E:000001': 'parity error', 'E:000002': 'input buffer overflow', 'E:000003': 'framing error',
          'E:000004': 'overrun', 'E:000010': 'CR or LF missing', 'E:000011': ': missing',
          'E:000012': 'invalid number of characters', 'E:000023': 'invalid value',
          'E:000030': 'value out of range', 'E:000041': 'command not applicable for this hardware',
          'E:000060': 'ZERO disabled', 'E:000080': 'command not accepted: valve in local operation',
          'E:000081': 'command not accepted: service interface locked',
          'E:000082': 'command not accepted: synchronization, interlock, safety mode or fatal error',
          'E:000089': 'not accepted in calibration and test mode'}

POSITION_MAX = 100000
UNKNOWN_POSITION = 999999

class ValveError(Exception):
    pass

def send(ser, cmd):
    # send one command, wait for the acknowledgement and check it
    ser.reset_input_buffer()
    ser.write((cmd + "\r\n").encode("ascii"))
    reply = ser.readline().decode("ascii", errors="replace").strip()
    if reply == "":
        raise ValveError(f"no answer to {cmd!r}")
    if reply.startswith("E:"):
        raise ValveError(f"{cmd!r} refused: {reply}, {ERRORS.get(reply, 'unknown error')}")
    expected = cmd if cmd.startswith("i:") or cmd == "A:" else cmd[:2]
    if not reply.startswith(expected):
        raise ValveError(f"unexpected answer {reply!r} to {cmd!r}")
    return reply

def get_status(ser):
    s = send(ser, "i:30")[4:]
    return {'access': ACCESS_MODES.get(s[0], s[0]),
            'mode': CONTROL_MODES.get(s[1], s[1]),
            'warning': s[3] == '1'}

def get_position(ser):
    p = int(send(ser, "A:")[2:])
    return None if p == UNKNOWN_POSITION else p

def set_position(ser, position):
    if not 0 <= position <= POSITION_MAX:
        raise ValueError(f"position must be between 0 and {POSITION_MAX}, not {position}")
    send(ser, f"R:{int(position):06d}")

In [3]:
def wait_for_position(ser, target, log, tolerance=100, timeout=30, poll=0.5, t0=0.0):
    start = time.time()
    while True:
        p = get_position(ser)
        log.append((time.time() - t0, p, get_status(ser)['mode']))
        if p is not None and abs(p - target) <= tolerance:
            return p
        if time.time() - start > timeout:
            raise ValveError(f"position {target} not reached after {timeout} s (last position: {p})")
        time.sleep(poll)

def run_sequence(ser, position, hold_time, tolerance=100, poll=0.5):
    log = []
    t0 = time.time()

    # 1. status
    status = get_status(ser)
    print("status:", status)
    if status['access'] == 'local':
        raise ValveError("valve in local operation, RS232 commands would be refused")
    if status['mode'] in ('interlock open', 'interlock closed', 'power failure', 'safety mode', 'fatal error'):
        raise ValveError(f"valve not ready: {status['mode']}")
    if status['warning']:
        print("warning:", send(ser, "i:51"))

    # 2. current position (unknown during synchronization)
    p = get_position(ser)
    while p is None:
        if time.time() - t0 > 30:
            raise ValveError("position still unknown after 30 s")
        time.sleep(1)
        p = get_position(ser)
    print(f"initial position: {p}")

    try:
        # 3. set the position
        set_position(ser, position)
        p = wait_for_position(ser, position, log, tolerance, poll=poll, t0=t0)
        print(f"t = {time.time()-t0:5.1f} s: position {p} reached")

        # 4. hold it
        send(ser, "H:")
        print(f"t = {time.time()-t0:5.1f} s: hold for {hold_time} s, mode = {get_status(ser)['mode']}")
        t_end = time.time() + hold_time
        while time.time() < t_end:
            time.sleep(min(poll, t_end - time.time()))
            log.append((time.time() - t0, get_position(ser), get_status(ser)['mode']))
    finally:
        # 5. close, even after an error
        send(ser, "C:")
        wait_for_position(ser, 0, log, tolerance=0, poll=poll, t0=t0)
        print(f"t = {time.time()-t0:5.1f} s: valve closed, mode = {get_status(ser)['mode']}")
    return log

Use with the real valve (30 % open for 60 s):

In [ ]:
for p in port_list.comports():
    print(p)

serial_address = "COM4"   # from the list above

with serial.Serial(serial_address, **VALVE_SETTINGS) as ser:
    log = run_sequence(ser, position=30000, hold_time=60)